# TCC2 - Previsao de Surtos de Dengue no Brasil
**Autores:** Pedro Lucas Santana e Thiago Ribeiro Freitas  
**Curso:** Engenharia de Software - UnB  

Este notebook carrega os dados do Google Drive e executa o pipeline de treinamento XGBoost.

**Estrutura esperada no Drive:**
```
My Drive/
  TCC2-DADOS/
    sinan/
    inmet/
    integrated/
    model_ready/
    reference/
```

**Repositorio GitHub:** [MODELO-PREVISAO](https://github.com/modelo-previsao-dengue/MODELO-PREVISAO)

---
## 1. Setup: Montar Drive e Instalar Dependencias

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

DATA_DIR = '/content/drive/MyDrive/TCC2-DADOS'

# Detecta se os dados estao direto em TCC2-DADOS/ ou dentro de TCC2-DADOS/data/
if os.path.exists(os.path.join(DATA_DIR, 'data', 'model_ready')):
    DATA_DIR = os.path.join(DATA_DIR, 'data')
    print(f'Dados encontrados em TCC2-DADOS/data/')
elif os.path.exists(os.path.join(DATA_DIR, 'model_ready')):
    print(f'Dados encontrados em TCC2-DADOS/')
else:
    raise FileNotFoundError(
        f'Pasta model_ready nao encontrada. Verifique se a pasta data/ foi enviada para TCC2-DADOS no Drive.'
    )

In [ ]:
!pip install -q xgboost optuna shap pyarrow

In [ ]:
import pandas as pd
import numpy as np
import pyarrow.parquet as pq
import pyarrow.dataset as ds
import xgboost as xgb
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    roc_auc_score, f1_score, confusion_matrix, classification_report
)
import matplotlib.pyplot as plt
import json
from pathlib import Path

print('Dependencias carregadas com sucesso.')

---
## 2. Verificar Dados no Drive

In [ ]:
base = Path(DATA_DIR)

print('=== Estrutura do Drive ===')
for folder in ['sinan', 'inmet', 'integrated', 'model_ready', 'reference']:
    p = base / folder
    if p.exists():
        size_mb = sum(f.stat().st_size for f in p.rglob('*') if f.is_file()) / 1e6
        n_files = sum(1 for f in p.rglob('*') if f.is_file())
        print(f'  {folder:15s}  {size_mb:8.1f} MB  ({n_files} arquivos)')
    else:
        print(f'  {folder:15s}  NAO ENCONTRADO')

---
## 3. Carregar Dataset Model-Ready

In [ ]:
MODEL_READY = base / 'model_ready'

train = pd.read_parquet(MODEL_READY / 'train.parquet')
val = pd.read_parquet(MODEL_READY / 'val.parquet')
test = pd.read_parquet(MODEL_READY / 'test.parquet')

print(f'Train: {len(train):>10,} linhas  (< 2022)')
print(f'Val:   {len(val):>10,} linhas  (2022-2023)')
print(f'Test:  {len(test):>10,} linhas  (2024+)')
print(f'Total: {len(train)+len(val)+len(test):>10,} linhas')
print(f'Features: {train.shape[1]} colunas')

In [ ]:
# Separar features e targets
ID_COLS = ['ibge_municipio', 'ano', 'semana_epidemiologica']
TARGET_REG = 'notificacoes_t4'
TARGET_CLF = 'risco_surto_t4'

feature_cols = [c for c in train.columns if c not in ID_COLS + [TARGET_REG, TARGET_CLF]]
print(f'{len(feature_cols)} features para modelagem')

X_train, y_train = train[feature_cols], train[TARGET_REG]
X_val, y_val = val[feature_cols], val[TARGET_REG]
X_test, y_test = test[feature_cols], test[TARGET_REG]

print(f'\nDistribuicao do target (notificacoes_t4):')
print(y_train.describe())

---
## 4. XGBoost Regressao (previsao t+4 semanas)

In [ ]:
model_reg = xgb.XGBRegressor(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method='hist',
    early_stopping_rounds=50,
    random_state=42,
    n_jobs=-1,
)

model_reg.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=50,
)

y_pred = np.maximum(model_reg.predict(X_test), 0)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f'\n=== Resultados Regressao (Test Set) ===')
print(f'RMSE: {rmse:.4f}')
print(f'MAE:  {mae:.4f}')
print(f'R2:   {r2:.4f}')

In [ ]:
# Feature importance top 20
importances = pd.Series(model_reg.feature_importances_, index=feature_cols)
top20 = importances.nlargest(20)

fig, ax = plt.subplots(figsize=(10, 8))
top20.sort_values().plot.barh(ax=ax)
ax.set_title('Top 20 Features - XGBoost Regressao')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()

---
## 5. XGBoost Classificacao (risco de surto)

In [ ]:
y_train_clf = train[TARGET_CLF]
y_val_clf = val[TARGET_CLF]
y_test_clf = test[TARGET_CLF]

print('Distribuicao das classes (train):')
print(y_train_clf.value_counts().sort_index())
print(f'\nClasses: 0=baixo, 1=medio, 2=alto, 3=surto')

In [ ]:
model_clf = xgb.XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method='hist',
    early_stopping_rounds=50,
    random_state=42,
    n_jobs=-1,
    objective='multi:softprob',
    num_class=4,
)

model_clf.fit(
    X_train, y_train_clf,
    eval_set=[(X_val, y_val_clf)],
    verbose=50,
)

y_pred_clf = model_clf.predict(X_test)
y_pred_proba = model_clf.predict_proba(X_test)

auc = roc_auc_score(y_test_clf, y_pred_proba, multi_class='ovr', average='macro')
f1 = f1_score(y_test_clf, y_pred_clf, average='macro')

print(f'\n=== Resultados Classificacao (Test Set) ===')
print(f'AUC macro: {auc:.4f}')
print(f'F1 macro:  {f1:.4f}')
print(f'\n{classification_report(y_test_clf, y_pred_clf, target_names=["baixo","medio","alto","surto"])}')

In [ ]:
# Confusion matrix
from sklearn.metrics import ConfusionMatrixDisplay

fig, ax = plt.subplots(figsize=(8, 6))
ConfusionMatrixDisplay.from_predictions(
    y_test_clf, y_pred_clf,
    display_labels=['baixo', 'medio', 'alto', 'surto'],
    ax=ax, cmap='Blues'
)
ax.set_title('Matriz de Confusao - Classificacao de Risco')
plt.tight_layout()
plt.show()

---
## 6. Baseline: SINAN-only vs SINAN+INMET

In [ ]:
# Identificar features climaticas
climate_cols = [c for c in feature_cols if any(k in c for k in [
    'temp_', 'rain_', 'humidity_', 'pressure_', 'wind_', 'radiation_'
])]
sinan_cols = [c for c in feature_cols if c not in climate_cols]

print(f'Features SINAN: {len(sinan_cols)}')
print(f'Features clima: {len(climate_cols)}')

# Treinar modelo SINAN-only
model_sinan = xgb.XGBRegressor(
    n_estimators=500, max_depth=6, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8, tree_method='hist',
    early_stopping_rounds=50, random_state=42, n_jobs=-1,
)
model_sinan.fit(
    train[sinan_cols], y_train,
    eval_set=[(val[sinan_cols], y_val)],
    verbose=0,
)

y_pred_sinan = np.maximum(model_sinan.predict(test[sinan_cols]), 0)

r2_full = r2_score(y_test, y_pred)
r2_sinan = r2_score(y_test, y_pred_sinan)
rmse_full = np.sqrt(mean_squared_error(y_test, y_pred))
rmse_sinan = np.sqrt(mean_squared_error(y_test, y_pred_sinan))

print(f'\n=== Comparacao ===')
print(f'{"Modelo":<20s} {"R2":>8s} {"RMSE":>10s}')
print(f'{"SINAN+INMET":<20s} {r2_full:>8.4f} {rmse_full:>10.4f}')
print(f'{"SINAN-only":<20s} {r2_sinan:>8.4f} {rmse_sinan:>10.4f}')
print(f'\n-> {"SINAN-only eh MELHOR" if r2_sinan > r2_full else "SINAN+INMET eh melhor"}')

---
## 7. SHAP - Interpretabilidade

In [ ]:
import shap

# Usar amostra para SHAP (50K linhas para nao estourar memoria)
sample_idx = np.random.RandomState(42).choice(len(X_test), min(50000, len(X_test)), replace=False)
X_sample = X_test.iloc[sample_idx]

explainer = shap.TreeExplainer(model_reg)
shap_values = explainer.shap_values(X_sample)

print('SHAP values calculados.')

In [ ]:
# Beeswarm plot
shap.summary_plot(shap_values, X_sample, max_display=20, show=True)

In [ ]:
# Top features por importancia SHAP
shap_importance = pd.DataFrame({
    'feature': feature_cols,
    'mean_abs_shap': np.abs(shap_values).mean(axis=0)
}).sort_values('mean_abs_shap', ascending=False)

shap_importance['is_climate'] = shap_importance['feature'].apply(
    lambda x: any(k in x for k in ['temp_', 'rain_', 'humidity_', 'pressure_', 'wind_', 'radiation_'])
)

print('Top 20 features por SHAP:')
print(shap_importance.head(20).to_string(index=False))

n_climate_top20 = shap_importance.head(20)['is_climate'].sum()
print(f'\nFeatures climaticas no top-20: {n_climate_top20}')

---
## 8. Explorar Dados Brutos (SINAN e INMET)

In [ ]:
# SINAN Gold - serie densa nacional
sinan_gold = ds.dataset(
    f'{DATA_DIR}/sinan/gold/sinan_tcc2_v2/official_dense',
    format='parquet', partitioning='hive'
)
print(f'SINAN Gold: {sinan_gold.count_rows():,} linhas')
print(f'Colunas: {[f.name for f in sinan_gold.schema][:10]}...')

# Carregar amostra
sinan_sample = sinan_gold.to_table().to_pandas().sample(5000, random_state=42)
print(f'\nAmostra SINAN Gold (5000 linhas):')
sinan_sample.head()

In [ ]:
# INMET Gold - features climaticas municipais
import glob

inmet_files = sorted(glob.glob(f'{DATA_DIR}/inmet/gold/weekly_municipal_climate_*.parquet'))
print(f'INMET Gold: {len(inmet_files)} arquivos anuais')

# Carregar um ano como exemplo
inmet_2024 = pd.read_parquet(inmet_files[-3])  # ~2024
print(f'\nINMET 2024: {len(inmet_2024):,} linhas, {inmet_2024.shape[1]} colunas')
print(f'Colunas: {list(inmet_2024.columns[:15])}...')
inmet_2024.head()

In [ ]:
# Cobertura INMET por ano
coverage = pd.read_csv(f'{DATA_DIR}/integrated/coverage_by_year.csv')

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(coverage['ano'], coverage['pct_coverage'], color='steelblue')
ax.axhline(y=50, color='red', linestyle='--', alpha=0.5, label='50%')
ax.set_xlabel('Ano')
ax.set_ylabel('Cobertura INMET (%)')
ax.set_title('Cobertura de Dados Climaticos por Ano (% municipios com dados INMET)')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Cobertura media: {coverage["pct_coverage"].mean():.1f}%')

---
## 9. Salvar Resultados de Volta no Drive

In [ ]:
# Criar pasta de resultados no Drive
results_dir = Path(DATA_DIR).parent / 'TCC2-RESULTADOS'
results_dir.mkdir(exist_ok=True)

# Salvar metricas
metrics = {
    'regressao': {'RMSE': round(rmse, 4), 'MAE': round(mae, 4), 'R2': round(r2, 4)},
    'classificacao': {'AUC_macro': round(auc, 4), 'F1_macro': round(f1, 4)},
    'comparacao': {
        'sinan_inmet_R2': round(r2_full, 4),
        'sinan_only_R2': round(r2_sinan, 4),
    }
}

with open(results_dir / 'metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

# Salvar modelo
model_reg.save_model(str(results_dir / 'xgb_regression.json'))
model_clf.save_model(str(results_dir / 'xgb_classification.json'))

print(f'Resultados salvos em {results_dir}')
print(f'Arquivos: {[f.name for f in results_dir.iterdir()]}')